In [2]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

In [8]:
# Load the dataset
df = pd.read_csv("/Reviews.csv")

# Remove rows with missing 'Text' values
df = df.dropna(subset=['Text'])

# Take a random sample of 1000 records for faster processing
df = df.sample(1000, random_state=42).reset_index(drop=True)
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,165257,B000EVG8J2,A1L01D2BD3RKVO,"B. Miller ""pet person""",0,0,5,1268179200,Crunchy & Good Gluten-Free Sandwich Cookies!,Having tried a couple of other brands of glute...
1,231466,B0000BXJIS,A3U62RE5XZDP0G,Marty,0,0,5,1298937600,great kitty treats,My cat loves these treats. If ever I can't fin...
2,427828,B008FHUFAU,AOXC0JQQZGGB6,Kenneth Shevlin,0,2,3,1224028800,COFFEE TASTE,A little less than I expected. It tends to ha...
3,433955,B006BXV14E,A3PWPNZVMNX3PA,rareoopdvds,0,1,2,1335312000,So the Mini-Wheats were too big?,"First there was Frosted Mini-Wheats, in origin..."
4,70261,B007I7Z3Z0,A1XNZ7PCE45KK7,Og8ys1,0,2,5,1334707200,Great Taste . . .,and I want to congratulate the graphic artist ...


In [4]:
def preprocess_text(text):
    doc = nlp(text.lower())  # lowercase and tokenize

    tokens = []
    for token in doc:
        # Keep only alphabetic tokens and remove stopwords
        if token.is_alpha and not token.is_stop:
            tokens.append(token.lemma_)  # use lemma (base form)
    return " ".join(tokens)

# Apply preprocessing to the 'Text' column
df['cleaned_text'] = df['Text'].apply(preprocess_text)


In [5]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['cleaned_text'])


In [6]:
def process_query(query):
    cleaned_query = preprocess_text(query)
    query_vec = vectorizer.transform([cleaned_query])
    return query_vec


In [7]:
def get_top_k_reviews(query, k=5):
    query_vec = process_query(query)

    # Calculate cosine similarity between query and all reviews
    cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get indices of top k similar reviews
    top_k_idx = cosine_similarities.argsort()[-k:][::-1]

    # Collect results
    results = []
    for idx in top_k_idx:
        results.append({
            "Review": df.loc[idx, "Text"],
            "Score": df.loc[idx, "Score"],
            "Similarity": cosine_similarities[idx]
        })
    return results


In [10]:
# Get and display top 5 reviews for a sample query
query = "great tasting coffee"
top_reviews = get_top_k_reviews(query)

for review in top_reviews:
    print(f"Review: {review['Review']}")
    print(f"Score: {review['Score']}")
    print(f"Similarity: {review['Similarity']:.4f}")
    print("-" * 20)

Review: Great coffee!  Love all Green Mountain coffee and all the wonderful flavors.  Would and do recommend this coffee to all my friends.
Score: 5
Similarity: 0.5553
--------------------
Review: This is very good coffee. I don't like strong coffee, this is nice and mild, a great breakfast blend. I use cream in it and it is just great. Will buy again.
Score: 5
Similarity: 0.4918
--------------------
Review: With hundreds (if not thousands) of coffee brands and/or types, it's tough to choose one, especially online without tasting.  I took a shot at Community Coffee's whole-bean French Roast.  I bought a Hamilton Beach coffee bean grinder, so I refuse to buy pre-ground coffee.  Psychologically, doing so would negate the value that the coffee-bean grinder would provide to me.<br /><br />With that aside, 2 of the 3 bags of coffee that came in the package were shrink-wrapped, meaning that you could see the shapes of the coffee beans through the package since it was vacuum sealed.  The othe